In [1]:
import sys
sys.path.append("../../utils")

import logging
import os
import warnings
from typing import List, Tuple
import torch
from sklearn.metrics import silhouette_score
import gseapy as gp
import anndata as ad
import pandas as pd
import scanpy as sc
from datasets import load_from_disk
from matplotlib.colors import Normalize
import squidpy as sq
from tqdm import tqdm
import pickle
import gc


# from nichejepa.utils.evaluation import (get_top_gene_score,
#                                         get_top_gene_pairs
#                                         )
#from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

import matplotlib.pyplot as plt

## This function takes a list of cell_ids and returns the corresponding dataset object.
def subset_by_cell_ids(dataset, cell_id_list):
    cell_id_set = set(cell_id_list)
    cell_ids = dataset['cell_id']
    indices = [i for i, cid in tqdm(enumerate(cell_ids), total=len(cell_ids), desc="Finding matching indices") if cid in cell_id_set]
    return dataset.select(indices)



/software/cellgen/team298/ls34/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team298/ls34/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team298/ls34/nichejepa/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from 

In [2]:
"""
note that app should be installed from the spatial FM - see https://github.com/Lotfollahi-lab/
"""
from app.infer import (embed_dataset,
                       harmonize_adata,
                       tokenize_adata,
                       perturb_dataset,
                       harmonize_tokenize_embed_pipeline,
                       get_gene_embed,
                       get_average_gene_embed,
                       get_spatial_score,
                       get_emd_distance )



In [3]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

sc.set_figure_params(
    dpi=50,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)
spot_size = 20

# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'



In [4]:
# Where model is located
model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/18062025_082526_412'
 
# SET PATH TO SAVE TOKENIZED ADATAS
# PATH_TO_TOKENIZED_ADATA = f'/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision/'
# PATH_TO_ADATA = '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/embedding/'
PATH_TO_TOKENIZED_ADATA = f'/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision_skinpanel/'
PATH_TO_ADATA = '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/embedding_allgenes_skinpanel/'

In [5]:
# min_cells_per_niche = 100
# latent_cluster_key = 'predicted_niche'


# Load dataset

In [6]:
# # adata_path='/nfs/team298/ls34/adult_skin/final_adatas/adata_xenium_freeze_plus3d.h5ad.september.plusnewdata'
# # adata_all=sc.read_h5ad(adata_path)
# PATH = '/nfs/team298/ls34/adult_skin/final_adatas/adata_all_and_newtime_mintflow.h5ad'
# #"/nfs/team298/ls34/adult_skin/final_adatas/adata_combined_new.h5ad.final.filtered"
# adata_all=sc.read_h5ad(PATH)
# #adata_all=adata_all[adata_all.obs["batch_nc"]=="query"].copy()
# adata_all.shape
adata_skinpanel=sc.read_h5ad('/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_output//SpatialSkinAtlasMapping_scanviSpatialSkinAtlasMapping_TUTORIAL_XENIUMskinpanel_HVGNUMBER259__MAXEPOCHS20__BATCHKEYsample_id/adata_TUTORIAL_XENIUMskinpanel+HVGNUMBER259__MAXEPOCHS20__BATCHKEYsample_id.h5ad')
adata_all=adata_skinpanel[adata_skinpanel.obs["Mapping_status"]!="SpatialSkinAtlas"].copy()


/software/cellgen/team298/ls34/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/software/cellgen/team298/ls34/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
# adata_all.shape

In [8]:
# KEEP = ['BK68-SKI-27-FO-1-S4-B2', 
#     'BK58-SKI-27-FO-4-S4-A2', 'BK58-SKI-27-FO-1-S5-A1', 'BK65-SKI-21-FO-1-S4-B1', 'BK36-SKI-27-FO-1-S5-E1', 'BK40-SKI-27-FO-1-S4-D2', 'BK44-SKI-27-FO-1-S4-E2', 'BK36-SKI-27-FO-1-S4-D1', 'BK37-SKI-27-FO-1-S4-C1', 'BK42-SKI-27-FO-1-S4-C2']
# adata_all = adata_all[adata_all.obs.info_id6.isin(KEEP)]
# adata_all.obs.info_id6.value_counts()

In [9]:
# adata_all.obs.info_id6.value_counts()

In [10]:
# "aaabclnl-1_output-XETG00055__0071160__BK58-SKI-27-FO-4-S4-A2_BK68-SKI-27-FO-1-S4-B2__20251204__150948" =='aaabclnl-1_output-XETG00055__0071160__BK58-SKI-27-FO-4-S4-A2_BK68-SKI-27-FO-1-S4-B2__20251204__150948'

In [11]:
# 'aaabclnl-1_output-XETG00055__0071160__BK58-SKI-27-FO-4-S4-A2_BK68-SKI-27-FO-1-S4-B2__20251204__150948' in adata_all.obs.index

In [12]:
# import pickle
# from pathlib import Path

# out = Path("/nfs/team298/ls34/dicts/final17_skin.pkl")

# with open(out, "rb") as f:
#     md = pickle.load(f)

In [13]:
# adata_all.obs["final17"] = adata_all.obs.index.map(md)
# adata_all.obs["final17"].value_counts()

In [14]:
9

9

In [15]:
# Attach ensembl IDs (gene names should be in .var_names)
file_path = '/nfs/team298/ls34/ensmbl_gene_5k.pkl'     # or the full path if you moved it

with open(file_path, "rb") as f:
    gene2ens = pickle.load(f)

if "ensembl_id" not in adata_all.var.columns:
    adata_all.var["ensembl_id"] = adata_all.var.index.map(gene2ens)
    adata_all.var["gene_name"] = adata_all.var.index
    adata_all.var.index=adata_all.var["gene_name"] 
    del(adata_all.var["gene_name"] )
adata_all.var.head()
adata_all.shape

(368818, 259)

In [16]:
# set cell type key
cell_type_key = 'scanvi_predictions'


# Tokenise + extract niche embeddings for all data 


In [17]:
"""
here, sample id's are in adata.obs["info_id6"]
so we will go through each sample, tokenise + save, then extract niche embeddings for that sample
"""

'\nhere, sample id\'s are in adata.obs["info_id6"]\nso we will go through each sample, tokenise + save, then extract niche embeddings for that sample\n'

In [18]:
emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None
top_k = None


In [19]:
# # LENGTH=adata_all.obs["info_id6"].unique()
# LENGTH

In [20]:
# LENGTH

In [21]:
PATH_TO_ADATA

'/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/embedding_allgenes_skinpanel/'

In [22]:
LENGTH=adata_all.obs["sample_id"].unique()
for i,SECTION in enumerate(LENGTH):
    print(SECTION)
    print(i, "/", len(LENGTH))
    dataset_name=SECTION
    save_dataset_path = PATH_TO_TOKENIZED_ADATA+ f'adata_{dataset_name}.h5ad'
    print(save_dataset_path)
    if not os.path.exists(PATH_TO_ADATA +  f'adata_{dataset_name}.h5ad'):
        adata=adata_all[adata_all.obs["sample_id"]==SECTION].copy()
        adata = harmonize_adata(adata)
        dataset = tokenize_adata(adata,
                             model_folder_path,
                             nproc = 4,
                             processing_mode = 'parallel')
        num_shards = 32
        dataset.save_to_disk(
                    save_dataset_path,
                    num_shards=num_shards)
    else:
        continue
#         print("load")
#         try:
#             dataset = load_from_disk(save_dataset_path)
            
#         except:
#             adata=sc.read_h5ad(save_dataset_path)
#             adata = harmonize_adata(adata)
#             dataset = tokenize_adata(adata,
#                                  model_folder_path,
#                                  nproc = 4,
#                                  processing_mode = 'parallel')
#             num_shards = 32
#             dataset.save_to_disk(
#                         save_dataset_path,
#                         num_shards=num_shards)
    output_embed = embed_dataset(
        dataset=dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        agg_excluded_tokens=agg_excluded_tokens,
        top_k=top_k,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers)
    # cell embedding (not used here)
    adata.obsm['cell_emb'] = output_embed['cell_emb']
    # niche embedding)
    adata.obsm['FM_niche_embedding'] = output_embed['neighborhood_emb']
    adata.write(PATH_TO_ADATA +  f'adata_{dataset_name}.h5ad')
    del(adata)
    gc.collect()
 

   

output-XETG00335__0032253__BK21-SKI-24-FO-1-S4__20240711__111911
0 / 16
/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision_skinpanel/adata_output-XETG00335__0032253__BK21-SKI-24-FO-1-S4__20240711__111911.h5ad
output-XETG00335__0032237__BK18-SKI-28-FO-1-S4__20240711__111911
1 / 16
/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision_skinpanel/adata_output-XETG00335__0032237__BK18-SKI-28-FO-1-S4__20240711__111911.h5ad
output-XETG00335__0032253__BK21-SKI-28-FO-2-S4__20240711__111911
2 / 16
/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision_skinpanel/adata_output-XETG00335__0032253__BK21-SKI-28-FO-2-S4__20240711__111911.h5ad
output-XETG00335__0032237__BK18-SKI-27-FO-1-S4__20240711__111911
3 / 16
/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/tokenized_adata_revision_skinpanel/adata_output-XETG00335__0032237__BK18-SKI-27-FO-1-S4__20240711__111911.h5ad
output-XETG00335__0032253__BK21-SKI-24-27-FO-2-1-S4__20240711__111911
4 / 16

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_l/ls34/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_l/ls34/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_l/ls34/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:nichejepa.tokenizers.cell_tokenizers:Loading token dictionary from /nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/18062025_082526_412/token_dictionary.pkl.


Number of genes with matching ensembl IDs:           256.
Number of genes skipped:           3.
STEP 2: ADDING SPECIAL VALUES...
STEP 1: LOADING CONFIG...
STEP 2: TOKENIZING ANNDATA OBJECT...
Filtering cells...
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood...
Normalizing gene expression counts...
Retrieving gene tokens.


/nfs/team361/ls34/nemo/nichejepa/src/nichejepa/tokenizers/cell_tokenizers.py:656: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var['ensembl_id'][coding_miRNA_idx]


Ranking gene tokens based on normalized counts.
Retrieving tokens for neighborhood cells.


...

INFO:nichejepa.tokenizers.cell_tokenizers:Creating Hugging Face dataset...
INFO:nichejepa.tokenizers.cell_tokenizers:Using dictionary for dataset creation.
INFO:nichejepa.tokenizers.cell_tokenizers:Formatting gene tokens...
Saving the dataset (32/32 shards): 100%|██████████| 12730/12730 [00:03<00:00, 3824.22 examples/s]


STEP 1: LOADING CONFIG...
STEP 2: GENERATING EMBEDDINGS...


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCountEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr', 'iter_number']


0it [00:00, ?it/s]/nfs/team361/ls34/nemo/nichejepa/src/app/infer.py:1014: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/software/cellgen/team298/ls34/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
100it [13:16,  7.96s/it]


# Plot niche embeddings

In [23]:
syop

NameError: name 'syop' is not defined

In [24]:
1

1

In [ ]:
1

In [ ]:
2

In [ ]:
2

In [ ]:
import scanpy as sc
import anndata as ad

import rapids_singlecell as rsc
import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
 
rmm.reinitialize(managed_memory=True, pool_allocator=False)
cp.cuda.set_allocator(rmm_cupy_allocator)


In [ ]:
9

In [ ]:
PATH_TO_TOKENIZED_ADATA= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/embedding/'
    
    

In [ ]:
import os
ADATAS = []

for fname in sorted(os.listdir(PATH_TO_TOKENIZED_ADATA)):
    if not fname.endswith(".h5ad"):
        continue

    path = os.path.join(PATH_TO_TOKENIZED_ADATA, fname)

    ad = sc.read_h5ad(path)

    # Keep provenance (ABSOLUTELY do this)
    ad.obs["source_file"] = fname

    ADATAS.append(ad)

# Concatenate into a single AnnData
adata = sc.concat(
    ADATAS,
    axis=0,                # cells
    join="outer",          # union of genes
    label="batch",         # adds adata.obs["batch"]
    keys=[a.obs["source_file"].iloc[0] for a in ADATAS],
    index_unique="-",      # avoid obs index collisions
)
import gc

del ADATAS
gc.collect()

In [ ]:
import gc

In [ ]:
adata

In [ ]:
adata_nemo=sc.read_h5ad('/nfs/team298/ls34/adult_skin/final_adatas/adata_nemofinal.h5ad.plots')
adata_nemo.obsm["FM_niche_embedding"]=adata_nemo.obsm["neighborhood_emb"]
adata_nemo

In [ ]:
adata = sc.concat(
    [adata, adata_nemo],
    #axis=0,                # cells
    join="outer",          # union of genes
    label="study",         # adds adata.obs["batch"]
    keys=["newnonresponse", "atlas"],
    #index_unique="-",      # avoid obs index collisions
)
import gc
gc.collect()

In [ ]:
# adata.X[:10,:10].A

In [ ]:
9

In [ ]:
rsc.pp.neighbors(adata,
                n_neighbors=10,
                use_rep='FM_niche_embedding',
                key_added='neighborhood')
rsc.tl.umap(adata,
           neighbors_key='neighborhood',
          min_dist=0.1)





In [ ]:
import pickle

with open("/nfs/team298/ls34/dicts/skin_niche19.pkl", "rb") as f:
    md = pickle.load(f)
adata.obs["niche19"] =adata.obs.index.map(md)
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche19")

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche19",
          groups=["Sebaceous_immune"],
           s=2
          )

In [ ]:
# Set dataset params
emb_key = 'neighborhood'
latent_leiden_resolution = 2
latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

rsc.tl.leiden(adata,
             neighbors_key=emb_key,
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution)


In [ ]:
adata.shape

In [ ]:
# adata.write(PATH_TO_ADATA +  f'adata_{dataset_name}.h5ad')


In [ ]:
adata

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche14")

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche14",
          groups=["Sebaceous_immune"],
           s=10
          )

In [ ]:
for x in adata.obs.columns:
    adata.obs[x]=    adata.obs[x].astype(str)

In [ ]:
adata.write('/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_nemo_all2.h5ad')
    

In [ ]:
adata

In [ ]:
99

In [ ]:
import scanpy as sc
import rapids_singlecell as rsc
import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
 
rmm.reinitialize(managed_memory=True, pool_allocator=False)
cp.cuda.set_allocator(rmm_cupy_allocator)


In [ ]:
adata=sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_nemo_all2.h5ad')
    

In [ ]:
latent_key = "neighbor_20"
#rsc.tl.leiden(adata, resolution=0.5, key_added="latent_leiden0.5", neighbors_key= "neighborhood" )
#rsc.tl.leiden(adata, resolution=2, key_added="latent_leiden2", neighbors_key= "neighborhood" )
rsc.tl.leiden(adata, resolution=3, key_added="latent_leiden3", neighbors_key= "neighborhood" )

import pickle

In [ ]:
import pickle

with open("/nfs/team298/ls34/dicts/skin_niche19.pkl", "rb") as f:
    md = pickle.load(f)
adata.obs["niche19"] =adata.obs.index.map(md)

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche19",
          #groups=["Sebaceous_immune"],
           s=2
          )

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="niche19",
          groups=["Sebaceous_immune"],
           s=2
          )

In [ ]:
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color="latent_leiden2",
          #groups=["Sebaceous_immune"],
           s=2
          )

In [ ]:
2

In [ ]:
RENAME = {}
GROUP = latent_cluster_key #"latent_leiden3"
ANNO_COL = "niche19"

adata_i = adata.copy()
for cluster in sorted(adata.obs[GROUP].unique()):
    # Subset the dataframe to that cluster
    #adata_i=adata_i[adata_i.obs["tech"]!="xenium"]
    #adata_i=adata_i[adata_i.obs[ANNO_COL]!=  'fibroblast progenitors PDGFRA+']
    #adata_i=adata_i[adata_i.obs[ANNO_COL].str.lower()!=  'fibroblast progenitors PDGFRA+']
   # adata_i=adata_i[adata_i.obs[ANNO_COL]!=  'FIBROBLAST PROGENITORS PDGFRA+']
    adata_i=adata_i[adata_i.obs[ANNO_COL]!=  'nan']

    adata_i=adata_i[adata_i.obs[ANNO_COL]!=  'reticulocyte']

    subset = adata_i.obs[adata_i.obs[GROUP] == cluster]
    # Count the values in combined_anno
    counts2 = subset[ANNO_COL].value_counts()
    
    # # Print top 3 contributors
    # print(f"Cluster {cluster} — Top 3 contributors:")
    # print(counts2.head(3), "\n")
    
    # Store the top contributor in RENAME
    if not counts2.empty:
        RENAME[cluster] = counts2.index[0]


        
adata.obs["provisional"]=adata.obs[GROUP].map(RENAME)
sc.pl.umap(
    adata,#[adata.obs["new_;ab"].isin(counts[counts >= 30].index)],
    color=[
         "provisional",  ANNO_COL
    ],
    legend_loc="on data",
    s=100, #neighbors_key="n",
    legend_fontsize=17,
    legend_fontoutline=2,
    # frame_on=False
 
)


In [ ]:
i=0
#adata_5k=adata_5k[adata_5k.obs["Site_status_binary"]=="Lesional"]
for tissue_section_id in adata[adata.obs["study"]=="newnonresponse"].obs.info_id6.unique():
    adata_i =  adata[adata.obs["info_id6"]==tissue_section_id] 
    sc.pl.spatial(
            adata_i,
            #library_id="spatial",
            #shape=None,
            color="provisional",
            spot_size=5,
            vmax=1,
        title=tissue_section_id,
           # title=STATUS + "_n=" + str(CELL_COUNT_IN_SECTION),
            #palette=custom_palette,
          #  save="perineural_mac_umap_NCIHE.pdf",
          edgecolor='black',
            linewidth=0.2 ,
            #ax=ax,
            #legend_loc="on data"  # Disable the legend for each subplot
        )
          
      

In [ ]:
1

In [ ]:
adata.obs["study"].value_counts()

In [ ]:
sc.set_figure_params(
    dpi=50,
    dpi_save=300,
    figsize=(30, 30),
    facecolor='white',
    fontsize=7
)
sc.pl.umap(
    adata,#[adata.obs["new_;ab"].isin(counts[counts >= 30].index)],
    color=[
         "provisional",  ANNO_COL
    ],
    legend_loc="on data",
    s=100, #neighbors_key="n",
    legend_fontsize=17,
    legend_fontoutline=2,
    # frame_on=False
 
)

In [ ]:
adata[adata.obs["study"]=="newnonresponse"].obs.info_id6.value_counts()

In [ ]:
adata.obs.provisional.value_counts()

In [ ]:
import pickle
from pathlib import Path

out = Path("/nfs/team298/ls34/dicts/final17_skin.pkl")

# Load the dict
with open(out, "rb") as f:
    md_loaded = pickle.load(f)


adata_i.obs["tmp"]=adata_i.obs.index.map(md_loaded)
adata_i.obs["Annotation3"]=adata_i.obs["tmp"].fillna("new")
adata_i.obs["Annotation3"].isna().sum()

In [ ]:
adata_i.obs.Annotation3.value_counts()

In [ ]:
import pickle

try:
    file_path = "/nfs/team298/ls34/dicts/color_for_adult_skin2.pkl"
    with open/(file_path, "rb") as f:
        colors = pickle.load(f)
except:
    !wget "https://raw.githubusercontent.com/haniffalab/spatial_skin_atlas/main/misc/color_for_adult_skin2.pkl" \
         -O color_for_adult_skin2.pkl
    file_path = "./color_for_adult_skin2.pkl"
    with open(file_path, "rb") as f:
        colors = pickle.load(f)
colors= colors | {'New/unlabelled/excluded': "#C8C8C8",
                  'New/unlabelled/unspecific':  "#C8C8C8",
                  'F1: Superficial': '#ffef5a',
    'F2/3: Perivascular': '#364f99',
    'F2: Universal': '#91bae2',
    'F3: FRC-like': '#c6508f',
    'F4: DP_HHIP+': '#c9efb4',
    'F4: DS_DPEP1+': '#3d6f3b',
    'F4: TNN+COCH+': '#00f273',
    'F5: NGFR+': '#8981a7',
    'F5: RAMP1+': '#4b2657',
     'F6: Inflammatory myofibroblast': "#75fbfd",
     'F7: Myofibroblast': "#2f6565",
       'F8: Fascia-like myofibroblast':"#EFA093", #'#dd7465',
 'F_Fascia': "#00004D",
                     "Th2": "#2A6072",      # Dark Blue (classic, strong contrast)
    "Tfh-like": "#AAA9A9",  # Neon Blue (bright, electric tone)
    "Th17": "#FFB3B3",
                  "adipocyte": '#B8860B'

     #   "TRM_IL13+": "#7D93C7"
    
                               } 
print(f"Loaded {len(colors)} colour entries")

colors2 = {'Sweat gland_clear': '#8AEAEA',
 'Sweat gland_dark': '#006D6D',

 'LE1': '#FFFF2B',

 'MacTREM2hi': '#E9E2F0',
 'LE2': '#D5D500',
 'Scar_VE/Peri': '#FF9B9B',
 'Mac2_MMP19hi': '#DDD3E9',
 'Mast cell_Active_ABCC4hi': '#FFB14D',
 'Mac2_HRH1+': '#D2C4E2',
 'KC3*': '#758BCC',
 'KC1/2_cycling': '#e0c5d2',
 'KC4/5': '#272c5f',
 'KC4*': '#9FC2CD',
 'KC_HFSC': '#D600FF',
 'T_CD4ex': '#FFE8B3',
            'ASIC2+ Mechanoreceptor': '#F1F1F1',
            'lowq': '#F1F1F1',
                       'new': '#F1F1F8',


 'ASIC2+ Mechanoreceptor2': '#F1F1F1',
 'Tfh-like2': '#AAA9A9',
 'ASIC1_4_KIF5C+': '#6F6F6F',
 'TRM_ex': '#FFF8E1',
 'Mast cell_prolif': '#FFA64D',
 'Th17/IFNGhi': '#FFA000',
 'Tc1_GIMAPhi': '#FFD06A',
 'KC_HF: ORS_KLK8+_SRGN+': '#E8FFF0',
 'ASIC2+ Mechanoreceptor1': '#D9D9D9',
 'Tfh-like1': '#AAB7C3',
 'Th_IFNGhi': '#FFB00A',
 'T DC Doublet': '#B38066',
 'SCC_KC': '#7B9299',
 'Melanoma': '#4E3323',
 'BCC+SCC_KC': '#576166',
 'Merkel_KC': '#FFE5E5'}
colors=colors|colors2
COLORS = {
    "ILC3+CCL1+PTGDS+": "#FF2F92",        # neon pink
    "ILC1_Prolif": "#FFD800",             # bright yellow
    "Tγδ": "#8B5A2B",                     # brown
    "CD4+_Prolif": "#C2B280",             # dark beige
    "CD4_IL13+ (was TRM_IL17+)": "#87CEEB",# sky blue
    "Th(ribosomal)": "#F2E6D8",            # very pale beige
    "Tc3_IFNGhi_CCL3+": "#E6D3B1",         # neon-ish beige
    "Tc1_BACH2+": "#E39A2E",               # orange beige
    "Th_PPARGhi_SLC10A1+TNFhi": "#9BB7D4", # bluish beige
    "Tc2_GZMH+": "#F28C28",                # orange
    "Tc3_IFNGhi_exhausted": "#FFE5C4",     # very pale orange
    "CD4_Tex_GZMK+IL10+PDCD1+": "#EFE2CF", # very pale beige
    "ILC3_Early": "#F6B7C6",               # pale pink
    "Th17_CCL20+ZBTB16+LAYN+": "#6FBF73",  # greenish
    "Th1_KLRG1+": "#D8C4A8",               # beige
    "Tc2_GZMH+TOX+": "#F3E6D3",             # very pale beige (distinct)
    "TRM_IL13+_2": "#3A6EA5",               # blue
    "Th1_KLRG1+GZMK+A2M+LYAR+": "#FFF2A6",  # pale yellow
    "Th17/Th1__IFNG+CXCL13+SOX5+": "#FFCC00", # bright yellow (distinct hue)
    "Th17_IL17F+": "#C23B22",               # reddish
    "ILC3_Prolif": "#B8A17D",               # dark beige
    "Tc1_BACH2+_TOXhi": "#9E8B6B",          # darker beige
    "TRM_exhausted": "#E6F0FA",             # very very pale blue
    "Th17_LAG3lo_IL23RhiBLK+": "#B22222",   # deep reddish
    "CD8+_naive": "#F5EDDF",                # very pale beige
    "TRM_IL13+_3_CD9+KRT7+": "#1E90FF",     # neon blue
    "TRM_IL13+_1": "#0B3C5D",               # dark blue
    "CD4 (was TRM_IL17+)": "#F9E1E1",       # very very pale red
    "Tfh-like1_ICA1+": "#9E9E9E",            # grey
    "CD4_IL17+ (was TRM_IL17+)": "#800020", # burgundy
    "TRM_CD4+_IL17+ITGA1+": "#F4A6A6",      # pastel red
    "TRM2_exhausted": "#EDF4FB",            # very pale blue (distinct)
    "TRM_CD4/CD8_mixed": "#A89272",         # dark beige
    "Th_PPARGhi_HLF+": "#8C7A5E",            # dark beige (darker)
    "CD8+_naive_exhausted_HAVCR2+": "#E8C2B3", # pinkish beige
    "Th2/mac": "#000000",                   # black
    
        "CD8+_Prolif": "#C2B280",          # dark beige
    "CD4_LOWQ(was TRM_IL17+)": "#D3D3D3",  # light grey
    "Th2_SLC10A1+TNFhi": "#7EC8E3",     # sky blue (clear, not pastel mush)
    "Th17_IL17A+IL17F+": "#4CAF50",     # grass green
    "Th17_IL17A+IL23R+": "#A5D6A7",     # pale green
    "T_Prolif_mixed": "#1B9E3C",        # intense green (distinct from Th17)
    "Th2_SATB1+PRDX2+": "#EEEEEE",      # very pale grey (still visible on white)
     "lowq_ithink":      "#EDE6C8",  # pale beige
    "Tlowq":            "#E2D3A4",  # beige
    "Th1_IFNG+":        "#FFD800",  # bright yellow
    "lowq_ilc":         "#F2F2F2",  # very pale grey
    "Tfh-like_Ex":      "#5A5A5A",  # dark grey
    "Th17_ZBTB16+":     "#B8E3B1",  # pastel green
    "ILC_Stress":       "#D9D9D9",  # light grey
    "Tc0 i think":      "#C7B299",  # dark-ish beige
         "lowq_ithink": "#4D4D4D",
    "lowq_mac": "#595959",
    "Pericyte1doublet": "#666666",
    "Tlowq": "#737373",
    "KC2/lowqmaybe": "#808080",
    "KC2_lowq": "#8C8C8C",
    "lowq_ilc": "#999999",
    "ILC_Stress": "#A6A6A6",
    "Mac_Prolif": "#B3B3B3",
    "Tfh-like_Ex": "#BFBFBF",
    "Th1_IFNG+": "#CCCCCC",
    "Th17_ZBTB16+": "#D9D9D9",
    "Tc0 i think": "#E6E6E6",
    "KC_HF: ORS_CLEC2A+KLK8+": "#F0F0F0",
}
colors=colors|COLORS



In [ ]:
sc.set_figure_params(
    dpi=50,
    dpi_save=300,
    figsize=(30, 30),
    facecolor='white',
    fontsize=7
)


i=0
#adata_5k=adata_5k[adata_5k.obs["Site_status_binary"]=="Lesional"]
for tissue_section_id in adata_all.obs.info_id6.unique():
    adata_i =  adata_all[adata_all.obs["info_id6"]==tissue_section_id] 
    sc.pl.spatial(
            adata_i,
            #library_id="spatial",
            #shape=None,
            color="final17",
            spot_size=10,
            vmax=1,
        title=tissue_section_id,
           # title=STATUS + "_n=" + str(CELL_COUNT_IN_SECTION),
            #palette=custom_palette,
          #  save="perineural_mac_umap_NCIHE.pdf",
          edgecolor='black',
            linewidth=0.2 ,
       # groups="TRM_IL13+",
        cmap="Reds", 
        palette=colors|{"KC2_stress": "#000000",
                   
                   "Lowq": "#000011",}
            #ax=ax,
            #legend_loc="on data"  # Disable the legend for each subplot
        )
          
      

In [ ]:
99

In [ ]:
# adata.write('/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_nemo_all.h5ad')
    